In [1]:
import pandas as pd
import pycountry
import wbdata

In [2]:
if "snakemake" not in globals():
        from _helpers_notebooks import mock_snakemake
        snakemake = mock_snakemake(
                "prepare_political_stability",
        )

### File paths

In [3]:
political_stability_fn = snakemake.input.political_stability
bus_locations_fn = snakemake.input.bus_locations

# Output
political_stability_clustered_fn = snakemake.output.political_stability

# config
config = snakemake.config

### Get Political Stability data

In [4]:
# Columns: Country, Score, Rank, Period
# Score is the political stability index (higher = more stable)
ps_raw = pd.read_csv(political_stability_fn)
ps_raw.head()


,Country,Score,Rank,Period
0,Liechtenstein,1.58,1,1996-2024
1,Micronesia,1.50,2,2003-2024
2,Tuvalu,1.47,3,2003-2024
3,San Marino,1.37,4,2003-2024
4,Bhutan,1.35,5,1996-2024


### Map country names to ISO-3 codes


In [5]:
country_name_corrections = {
    # Config region list corrections (standard name → pycountry name)
    "Democratic Republic of the Congo": "Congo, The Democratic Republic of the",
    "Republic of the Congo": "Republic of the Congo",
    "Kosovo": "Republic of Kosovo",
    "Russia": "Russian Federation",
    "Turkey": "Türkiye",
    "Venezuela": "Venezuela, Bolivarian Republic of",
    "Tanzania": "United Republic of Tanzania",
    "Bolivia": "Plurinational State of Bolivia",
    "Vietnam": "Viet Nam",
    "South Korea": "Korea, Republic of",
    "North Korea": "Korea, Democratic People's Republic of",
    "Taiwan": "Taiwan, Province of China",
    "Laos": "Lao People's Democratic Republic",
    "Brunei": "Brunei Darussalam",
    "Equatorial French Guiana": "French Guiana",
    "Syria": "Syrian Arab Republic",
    "Palestine": "Palestine, State of",
    "Moldova": "Republic of Moldova",
    # globaleconomy.csv abbreviated / non-standard names → pycountry names
    "Micronesia":          "Micronesia, Federated States of",
    "N. Caledonia":        "New Caledonia",
    "Ant.& Barb.":         "Antigua and Barbuda",
    "St. Vincent & ...":   "Saint Vincent and the Grenadines",
    "UA Emirates":         "United Arab Emirates",
    "Cape Verde":          "Cabo Verde",
    "Solomon Isl.":        "Solomon Islands",
    "Domin. Rep.":         "Dominican Republic",
    "S.T.&Principe":       "Sao Tome and Principe",
    "Tr.&Tobago":          "Trinidad and Tobago",
    "UK":                  "United Kingdom",
    "USA":                 "United States",
    "Eq. Guinea":          "Equatorial Guinea",
    "R. of Congo":         "Congo",
    "Bosnia & Herz.":      "Bosnia and Herzegovina",
    "G.-Bissau":           "Guinea-Bissau",
    "Papua N.G.":          "Papua New Guinea",
    "Swaziland":           "Eswatini",
    "Ivory Coast":         "Côte d'Ivoire",
    "Iran":                "Iran, Islamic Republic of",
    "C.A. Republic":       "Central African Republic",
    "Burma":               "Myanmar",
    "DR Congo":            "Congo, The Democratic Republic of the",
}

# Build country name → ISO-3 lookup from pycountry
country_name_to_iso = {}
for country in pycountry.countries:
    country_name_to_iso[country.name] = country.alpha_3
    if hasattr(country, "official_name"):
        country_name_to_iso[country.official_name] = country.alpha_3
# Extra common-name aliases
country_name_to_iso["United States"] = "USA"
country_name_to_iso["United Kingdom"] = "GBR"
country_name_to_iso["Republic of the Congo"] = "COG"
country_name_to_iso["Republic of Kosovo"] = "XKX"

# Apply name corrections and map to ISO-3
ps = ps_raw.copy()
ps["Country"] = ps["Country"].replace(country_name_corrections)
ps["country_code"] = ps["Country"].map(country_name_to_iso)

n_unmatched = ps["country_code"].isna().sum()
print(f"Matched {ps['country_code'].notna().sum()} / {len(ps)} countries to ISO-3 codes")
if n_unmatched:
    print("Still unmatched:")
    print(ps[ps["country_code"].isna()]["Country"].tolist())

ps = ps.dropna(subset=["country_code"]).rename(columns={"Score": "political_stability"})
ps = ps[["country_code", "political_stability"]]


Matched 194 / 194 countries to ISO-3 codes


### Cluster per region

Map each country in the data to its region using the region definitions from the config file.

In [6]:
# Load region definitions from config (regions now uses ISO-3 codes directly)
regions = config["regions"]

# Build ISO-3 → region mapping
iso_to_region = {}
for region, iso_codes in regions.items():
    for code in iso_codes:
        iso_to_region[code] = region

In [7]:
# Assign region to each country
ps["region"] = ps["country_code"].map(lambda iso: iso_to_region.get(iso, "Other"))
print(ps["region"].value_counts())


region
Other                  37
Europe                 33
North_West_Africa      23
Subsaharan_Africa      21
Middle_East            13
Pacific_Asia           11
Central_America        11
Eurasia                 9
South_America           9
West_Asia               7
Far_West_Europe         6
East_Asia               4
South_South_America     3
East_East_Asia          3
Oceania                 2
North_America           2
Name: count, dtype: int64


### Cluster: Weighted by GDP


In [8]:
# --- fetch GDP (current USD) from World Bank ---
gdp_raw = wbdata.get_dataframe(
    {"NY.GDP.MKTP.CD": "gdp_usd"},
    date="2024",
)
gdp = gdp_raw.dropna(subset=["gdp_usd"]).reset_index()
gdp = gdp.rename(columns={"country": "country_name"})

# Map country name → ISO-3 so we can merge on ps["country_code"]
gdp["country_code"] = gdp["country_name"].map(lambda name: country_name_to_iso.get(name))
gdp = gdp.dropna(subset=["country_code"])

# --- attach GDP weights ---
ps_gdp = ps.merge(gdp[["country_code", "gdp_usd"]], on="country_code", how="left")

# Fill missing GDP with regional median so no country is dropped
ps_gdp["gdp_usd"] = ps_gdp.groupby("region")["gdp_usd"].transform(
    lambda x: x.fillna(x.median())
)

def weighted_mean(df, value_col, weight_col):
    return (df[value_col] * df[weight_col]).sum() / df[weight_col].sum()

ps_by_region = (
    ps_gdp.groupby("region")
    .apply(lambda g: pd.Series({
        "political_stability": weighted_mean(g, "political_stability", "gdp_usd"),
        "n_countries":         len(g),
        "ps_spread":           g["political_stability"].max() - g["political_stability"].min(),
    }))
    .reset_index()
    .set_index("region")
)

/tmp/ipykernel_20946/2609092797.py:26: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



In [9]:

ps_by_region.sort_values("political_stability", ascending=False)

,political_stability,n_countries,ps_spread
region,,,
Oceania,0.836432,2.0,0.36
Other,0.728116,37.0,2.31
East_East_Asia,0.696667,3.0,0.81
Europe,0.346611,33.0,2.04
East_Asia,0.312861,4.0,1.07
Far_West_Europe,0.088348,6.0,1.45
South_South_America,0.033099,3.0,1.45
North_America,-0.049328,2.0,0.70
Pacific_Asia,-0.141959,11.0,3.32


In [10]:
# Apply manual stability penalties from config (design.stability_penalty)
# Each entry subtracts the given value from the region's computed political_stability score.
stability_penalties = config.get("design", {}).get("stability_penalty", {})

if stability_penalties:
    for region, penalty in stability_penalties.items():
        if region in ps_by_region.index:
            original = ps_by_region.loc[region, "political_stability"]
            ps_by_region.loc[region, "political_stability"] -= penalty
            print(f"Applied penalty to {region}: {original:.4f} → {ps_by_region.loc[region, 'political_stability']:.4f} (−{penalty})")
        else:
            print(f"Warning: region '{region}' in stability_penalty not found in ps_by_region index.")
else:
    print("No stability penalties configured.")

ps_by_region.sort_values("political_stability", ascending=False)


Applied penalty to Middle_East: -0.5512 → -1.1512 (−0.6)


,political_stability,n_countries,ps_spread
region,,,
Oceania,0.836432,2.0,0.36
Other,0.728116,37.0,2.31
East_East_Asia,0.696667,3.0,0.81
Europe,0.346611,33.0,2.04
East_Asia,0.312861,4.0,1.07
Far_West_Europe,0.088348,6.0,1.45
South_South_America,0.033099,3.0,1.45
North_America,-0.049328,2.0,0.70
Pacific_Asia,-0.141959,11.0,3.32


### Save clustered Political Stability data

In [11]:
ps_by_region.to_csv(political_stability_clustered_fn)